In [1]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
import os

builder = SparkSession.builder \
    .appName("ETL Gold Layer") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
silver_path = "../delta_lake/silver/"

dim_artists = spark.read.format("delta").load(os.path.join(silver_path, "dim_artists"))
dim_albums = spark.read.format("delta").load(os.path.join(silver_path, "dim_albums"))
dim_genres = spark.read.format("delta").load(os.path.join(silver_path, "dim_genres"))
dim_tracks = spark.read.format("delta").load(os.path.join(silver_path, "dim_tracks"))
fact_tracks = spark.read.format("delta").load(os.path.join(silver_path, "fact_tracks"))

In [3]:
gold_path = "../delta_lake/gold/"

In [4]:
metrics = [
    "popularity", "danceability", "energy", "speechiness",
    "acousticness", "instrumentalness", "liveness", "valence"
]

## All info

In [6]:
# join fact_tracks with dim_tracks to get track_name, album_id, genre_id
fact_with_info = fact_tracks.join(
    dim_tracks.select("track_id", "track_name", "album_id", "genre_id"),
    on="track_id",
    how="inner"
).join(
    # join with dim_albums to get album_name and artist_id
    dim_albums.select("album_id", "album_name", "artist_id"),
    on="album_id",
    how="inner"
).join(
    # join with dim_artists to get artist name
    dim_artists.select("artist_id", "artists"),
    on="artist_id",
    how="inner"
).join(
    # join with dim_genres to get genre name
    dim_genres.select("genre_id", "track_genre"),
    on="genre_id",
    how="inner"
)

# album_name, track_name, artists, track_genre, and all fact metrics (excluding IDs)
columns_to_select = (
    ["album_name", "track_name", "artists", "track_genre"] +
    [c for c in fact_with_info.columns if c not in [
        "track_id", "album_id", "artist_id", "genre_id",
        "track_name", "album_name", "artists", "track_genre"
    ]]
)

all_info = fact_with_info.select(*columns_to_select)

all_info.show(10, truncate=False)

all_info.write.format("delta").mode("overwrite").save(os.path.join(gold_path, "all_info"))


+------------------------------------------------------+--------------------------+------------------------------------+-----------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|album_name                                            |track_name                |artists                             |track_genre|popularity|duration_ms|explicit|danceability|energy|loudness|tempo  |speechiness|acousticness|instrumentalness|liveness|valence|mode|key|time_signature|
+------------------------------------------------------+--------------------------+------------------------------------+-----------+----------+-----------+--------+------------+------+--------+-------+-----------+------------+----------------+--------+-------+----+---+--------------+
|Comedy                                                |Comedy                    |Gen Hoshino                         |acoustic   |73        |23

## Avg metrics per album

In [6]:
from pyspark.sql.functions import avg

avg_metrics_by_album_artist = (
    fact_tracks.alias("fact")
    .join(dim_tracks.alias("dt"), on="track_id")
    .join(dim_albums.alias("da"), on="album_id")
    .join(dim_artists.alias("ar"), on="artist_id")
    .groupBy("da.album_name", "ar.artists")
    .agg(*[avg(f"fact.{c}").alias(f"avg_{c}") for c in metrics])
)

avg_metrics_by_album_artist.show(5, truncate=False)

avg_metrics_by_album_artist.write.format("delta").mode("overwrite").save(gold_path + "avg_metrics_by_album_artist")

+-------------------------------------+--------------------------+--------------+------------------+-------------------+-------------------+-------------------+--------------------+-------------------+------------------+
|album_name                           |artists                   |avg_popularity|avg_danceability  |avg_energy         |avg_speechiness    |avg_acousticness   |avg_instrumentalness|avg_liveness       |avg_valence       |
+-------------------------------------+--------------------------+--------------+------------------+-------------------+-------------------+-------------------+--------------------+-------------------+------------------+
|pov: you have a holly jolly christmas|Albert King               |0.0           |0.5439999999999999|0.46048484848484855|0.0669909090909091 |0.5851848484848484 |7.417727272727272E-5|0.17483333333333334|0.6198787878787879|
|Abra Sua Cabeça                      |Abayomy Afrobeat Orquestra|22.5          |0.696             |0.88650000000000

## Avg metrics per artist

In [ ]:
fact_with_artists = (
    fact_tracks
    .join(dim_tracks.select("track_id", "album_id"), on="track_id", how="inner")
    .join(dim_albums.select("album_id", "artist_id"), on="album_id", how="inner")
    .join(dim_artists.select("artist_id", "artists"), on="artist_id", how="inner")
)

avg_metrics_by_artist = (
    fact_with_artists
    .groupBy("artists")
    .agg(*[avg(c).alias(f"avg_{c}") for c in metrics])
)

avg_metrics_by_artist.show(5, truncate=False)

avg_metrics_by_artist.write.format("delta").mode("overwrite").save(os.path.join(gold_path, "avg_metrics_by_artist"))

+-------------------------+------------------+------------------+------------------+--------------------+-------------------+---------------------+-------------------+-------------------+
|artists                  |avg_popularity    |avg_danceability  |avg_energy        |avg_speechiness     |avg_acousticness   |avg_instrumentalness |avg_liveness       |avg_valence        |
+-------------------------+------------------+------------------+------------------+--------------------+-------------------+---------------------+-------------------+-------------------+
|Boyce Avenue;Megan Nicole|53.42857142857143 |0.5172857142857143|0.3868571428571429|0.03137142857142857 |0.642              |1.6900000000000001E-6|0.11895714285714286|0.28574285714285713|
|Ramshackle Glory         |27.666666666666668|0.5416666666666666|0.437             |0.044366666666666665|0.28465999999999997|3.6333333333333334E-6|0.1322333333333333 |0.7400000000000001 |
|Brendan James            |51.0              |0.636         

## Avg metrics per genre

In [ ]:
fact_with_genre = (
    fact_tracks
    .join(dim_tracks.select("track_id", "genre_id"), on="track_id", how="inner")
    .join(dim_genres.select("genre_id", "track_genre"), on="genre_id", how="inner")
)

avg_metrics_by_genre = (
    fact_with_genre
    .groupBy("track_genre")
    .agg(*[avg(c).alias(f"avg_{c}") for c in metrics])
)

avg_metrics_by_genre.show(5, truncate=False)

avg_metrics_by_genre.write.format("delta").mode("overwrite").save(os.path.join(gold_path, "avg_metrics_by_genre"))

+-----------------+------------------+------------------+-------------------+--------------------+--------------------+---------------------+-------------------+-------------------+
|track_genre      |avg_popularity    |avg_danceability  |avg_energy         |avg_speechiness     |avg_acousticness    |avg_instrumentalness |avg_liveness       |avg_valence        |
+-----------------+------------------+------------------+-------------------+--------------------+--------------------+---------------------+-------------------+-------------------+
|anime            |48.776884422110555|0.5376657286432159|0.6742294472361817 |0.08742884422110551 |0.26711691732663356 |0.2625900097386936   |0.19748412060301548|0.4346443216080403 |
|singer-songwriter|43.59203036053131 |0.5502770398481973|0.41736755218216315|0.06069981024667933 |0.5651863036053135  |0.025600465863377615 |0.14853263757115737|0.3713546489563562 |
|folk             |40.1640903686088  |0.5624316290130787|0.5580824019024966 |0.06183091557

## Number of songs by each artist

In [ ]:
from pyspark.sql.functions import count

tracks_with_artists = (
    dim_tracks
    .join(dim_albums.select("album_id", "artist_id"), on="album_id", how="inner")
    .join(dim_artists, on="artist_id", how="inner")
)

song_count_by_artist = (
    tracks_with_artists
    .groupBy("artists")
    .agg(count("track_id").alias("song_count"))
)

song_count_by_artist.show(10, truncate=False)

song_count_by_artist.write.format("delta").mode("overwrite").save(os.path.join(gold_path, "song_count_by_artist"))

+---------------------------+----------+
|artists                    |song_count|
+---------------------------+----------+
|Boyce Avenue;Megan Nicole  |7         |
|Ramshackle Glory           |3         |
|Brendan James              |1         |
|The Black Keys             |14        |
|Yann Tiersen               |10        |
|Fabrizio Paterlini         |14        |
|Hugar                      |3         |
|Anime Piano Dreamers       |1         |
|Yonder Mountain String Band|21        |
|South Austin Jug Band      |1         |
+---------------------------+----------+
only showing top 10 rows

